In [ ]:
!pip install -q -U langgraph langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.7/571.7 kB 31.3 MB/s eta 0:00:00


In [ ]:
import os
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# 1. Define the Graph State schema
class AgentState(TypedDict):
    user_query: str
    response: str
    is_valid: bool

In [ ]:
# 2. Define Node Functions
def generate_response_node(state: AgentState):
    query = state["user_query"]
    # Simulating LLM output generation
    simulated_answer = f"Processed request: {query.upper()}"
    return {"response": simulated_answer}
def validation_node(state: AgentState):
    # Validates if response meets basic length requirements
    valid = len(state["response"]) > 10
    return {"is_valid": valid}

In [ ]:
#3. Define Routing Logic (Conditional Edge)
def router(state: AgentState):
    if state["is_valid"]:
        return "approved"
    else:
        return "rejected"

In [ ]:
# 4. Build the LangGraph State Machine
builder = StateGraph(AgentState)

# Add Nodes
builder.add_node("generator", generate_response_node)
builder.add_node("validator", validation_node)

# Wire Edges
builder.add_edge(START, "generator")
builder.add_edge("generator", "validator")

# Conditional Routing
builder.add_conditional_edges(
    "validator",
    router,
    {
        "approved": END,
        "rejected": "generator"  # Loops back if validation fails!
    }
)

app = builder.compile()

In [ ]:
# 5. Execute Graph
output = app.invoke({"user_query": "hello langgraph agent"})
print("--- LANGGRAPH EXECUTION COMPLETE ---")
print(f"Final State: {output}")

--- LANGGRAPH EXECUTION COMPLETE ---
Final State: {'user_query': 'hello langgraph agent', 'response': 'Processed request: HELLO LANGGRAPH AGENT', 'is_valid': True}
